In [1]:
from scipy.io import loadmat

data = loadmat(
    "raw_data/Data_Preprocessed_P01/Data_Preprocessed_P01.mat"
)

print(data.keys())

print("joined_data:", data["joined_data"].shape)
print("labels_selfassessment:", data["labels_selfassessment"].shape)

print(data["joined_data"][0,0].shape)
print(data["labels_selfassessment"][0,0])

dict_keys(['__header__', '__version__', '__globals__', 'VideoIDs', 'joined_data', 'labels_ext_annotation', 'labels_selfassessment'])
joined_data: (1, 20)
labels_selfassessment: (1, 20)
(12225, 17)
[[2.938568 5.       8.235496 1.       1.       1.       0.       0.
  0.       0.       0.       0.      ]]


In [2]:
import numpy as np

def create_windows(signal, window_size=128, step_size=64):

    windows = []

    for start in range(
        0,
        len(signal) - window_size + 1,
        step_size
    ):
        end = start + window_size

        windows.append(signal[start:end])

    return windows

In [3]:
import numpy as np

def extract_statistical_features(window):

    features = []

    for ch in range(window.shape[1]):

        signal = window[:, ch]

        features.extend([
            np.mean(signal),
            np.std(signal),
            np.min(signal),
            np.max(signal),
            np.ptp(signal),
            np.var(signal),
            np.sqrt(np.mean(signal**2))
        ])

    return features

In [4]:
from scipy.signal import welch
import numpy as np

def bandpower(freqs, psd, low, high):

    idx = np.logical_and(freqs >= low,
                         freqs <= high)

    return np.trapezoid(psd[idx], freqs[idx])

In [5]:
def extract_psd_features(window, fs=128):

    features = []

    for ch in range(window.shape[1]):

        signal = window[:, ch]

        freqs, psd = welch(
            signal,
            fs=fs,
            nperseg=min(len(signal), fs)
        )

        delta = bandpower(freqs, psd, 0.5, 4)
        theta = bandpower(freqs, psd, 4, 8)
        alpha = bandpower(freqs, psd, 8, 13)
        beta  = bandpower(freqs, psd, 13, 30)
        gamma = bandpower(freqs, psd, 30, 45)

        features.extend([
            delta,
            theta,
            alpha,
            beta,
            gamma
        ])

    return features

In [6]:
import os

dataset_dir = "raw_data"

participants = []

for folder in os.listdir(dataset_dir):

    if folder.startswith("Data_Preprocessed_"):
        folder = folder.replace("Data_Preprocessed_","")
        participants.append(folder)

print(participants)
print("Total Participants:", len(participants))

['P01', 'P07', 'P08', 'P12', 'P13', 'P14', 'P15', 'P22', 'P23', 'P24', 'P25', 'P28', 'P36', 'P37', 'P40']
Total Participants: 15


In [7]:
all_rows = []

for participant in participants:

    mat_file = os.path.join(
        dataset_dir,
        f"Data_Preprocessed_{participant}",
        f"Data_Preprocessed_{participant}.mat"
    )

    print(f"Processing {participant}")

    data = loadmat(mat_file)

    joined = data["joined_data"]
    labels = data["labels_selfassessment"]

    for video_idx in range(joined.shape[1]):

        signal = joined[0, video_idx]

        label = labels[0, video_idx][0]

        windows = create_windows(
            signal,
            window_size=128,
            step_size=64
        )

        for window in windows:

            stats = extract_statistical_features(window)
            psd = extract_psd_features(window)

            row = [
                participant,
                video_idx + 1,
                *label,
                *stats,
                *psd
            ]

            all_rows.append(row)

    print(f"Rows so far: {len(all_rows)}")

Processing P01
Rows so far: 12534
Processing P07
Rows so far: 25068
Processing P08
Rows so far: 27978
Processing P12
Rows so far: 40512
Processing P13
Rows so far: 53046
Processing P14
Rows so far: 65580
Processing P15
Rows so far: 78114
Processing P22
Rows so far: 90648
Processing P23
Rows so far: 103182
Processing P24
Rows so far: 106092
Processing P25
Rows so far: 118626
Processing P28
Rows so far: 121536
Processing P36
Rows so far: 134070
Processing P37
Rows so far: 146604
Processing P40
Rows so far: 159138


In [8]:
print("Total Rows:", len(all_rows))
print("Columns per Row:", len(all_rows[0]))

Total Rows: 159138
Columns per Row: 218


In [9]:
columns = (
    ["Subject_ID", "Video_ID"] +
    [f"Label_{i}" for i in range(12)] +
    [f"Feature_{i}" for i in range(204)]
)

import pandas as pd

df = pd.DataFrame(all_rows, columns=columns)

print(df.shape)

for i in range(12):
    print(f"\nLabel_{i}")
    print("Unique:", df[f"Label_{i}"].nunique())
    print(df[f"Label_{i}"].describe())

(159138, 218)

Label_0
Unique: 183
count    159138.000000
mean          5.919644
std           1.382339
min           1.218432
25%           5.000000
50%           6.280000
75%           6.680000
max           9.000000
Name: Label_0, dtype: float64

Label_1
Unique: 182
count    159138.000000
mean          5.255640
std           1.747007
min           1.000000
25%           3.880000
50%           5.080000
75%           6.680000
max           9.000000
Name: Label_1, dtype: float64

Label_2
Unique: 200
count    159138.000000
mean          4.580009
std           1.673725
min           1.000000
25%           3.320000
50%           4.120000
75%           6.040000
max           9.000000
Name: Label_2, dtype: float64

Label_3
Unique: 189
count    159138.000000
mean          6.272122
std           2.115343
min           1.000000
25%           5.000000
50%           6.788400
75%           7.800000
max           9.160000
Name: Label_3, dtype: float64

Label_4
Unique: 134
count    159138.000000
me

In [10]:
df["Target"] = (df["Label_0"] >= 5).astype(int)

feature_cols = [c for c in df.columns if c.startswith("Feature_")]

df_clean = df.dropna(subset=feature_cols)

print(df.shape)
print(df_clean.shape)

(159138, 219)
(153540, 219)


In [11]:
X = df_clean[feature_cols]
y = df_clean["Target"]
groups = df_clean["Subject_ID"]

In [12]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("X_train.shape = ",X_train.shape)
print("y_train.shape = ",y_train.shape)
print("X_test.shape = ",X_test.shape)
print("y_test.shape = ",y_test.shape)


X_train.shape =  (136406, 204)
y_train.shape =  (136406,)
X_test.shape =  (17134, 204)
y_test.shape =  (17134,)


In [13]:
from sklearn.preprocessing import StandardScaler
import joblib

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

filepath = os.path.join("Models/Preprocessor/scaler.pkl")
os.makedirs(os.path.dirname(filepath),exist_ok=True)

joblib.dump(scaler, filepath)

['Models/Preprocessor/scaler.pkl']

In [14]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

def evaluate_trained_model(
    model,
    model_name,
    X_test,
    y_test
):

    y_pred = model.predict(X_test)

    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average="weighted"),
        "Recall": recall_score(y_test, y_pred, average="weighted"),
        "F1 Score": f1_score(y_test, y_pred, average="weighted")
    }

In [15]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.svm import LinearSVC

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

sss = StratifiedShuffleSplit(
    n_splits=1,
    train_size=20000,
    random_state=42
)

svm_idx, _ = next(sss.split(X_train, y_train))

X_train_svm = X_train[svm_idx]
y_train_svm = y_train.iloc[svm_idx]

svm = LinearSVC(
    C=1.0,
    random_state=42,
    max_iter=10000
)

svm.fit(X_train_svm, y_train_svm)
rf.fit(X_train, y_train)

results = []

results.append(
    evaluate_trained_model(
        rf,
        "Random Forest",
        X_test,
        y_test
    )
)

results.append(
    evaluate_trained_model(
        svm,
        "Linear SVM",
        X_test,
        y_test
    )
)

comparison_df = pd.DataFrame(results)
comparison_df

,Model,Accuracy,Precision,Recall,F1 Score
0,Random Forest,0.800981,0.702545,0.800981,0.740388
1,Linear SVM,0.783880,0.724143,0.783880,0.747062
